# Independent 3D Gaussian Splatting implementation

This notebook documents my bachelor's thesis implementation. COLMAP loading, Gaussian parameters, training, adaptive density control and evaluation are implemented here. Rendering uses the original authors' CUDA rasterizer (`diff_gaussian_rasterization`). For setup, data layout and the runnable script, see [README.md](README.md).

## Step 0: Necessary Imports

In [ ]:
### basics

import os
import gc
import math
import random
import struct
import warnings

# Densification reallocates six large tensors (+ their Adam moments) every 100
# iterations, which is close to a worst case for the caching allocator.
# Must be set before torch initialises CUDA, hence before `import torch`.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

### base
import numpy as np
import matplotlib.pyplot as plt

### torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import time
import imageio
import lpips
from PIL import Image
from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors
from pytorch_msssim import ssim

from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer

warnings.filterwarnings("ignore", category=UserWarning)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# fused-ssim is one kernel instead of the ~6 convolutions pytorch_msssim runs.
# `pip install fused-ssim`; falls back silently if absent.
try:
    from fused_ssim import fused_ssim as _fused_ssim
    def ssim_loss(a, b):
        return _fused_ssim(a.unsqueeze(0), b.unsqueeze(0))
except ImportError:
    def ssim_loss(a, b):
        return ssim(a.unsqueeze(0), b.unsqueeze(0), data_range=1.0, size_average=True)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
### scene / protocol
SCENE      = "bonsai"   # indoor: room counter kitchen bonsai | outdoor: bicycle flowers garden stump treehill
DOWNSAMPLE = 2          # 2 for indoor, 4 for outdoor -- the mip-NeRF360 protocol
ITERATIONS = 30_000
SEED       = 42
TEST_EVERY = 8          # every 8th image is a held-out test view

### optimization (paper values)
POSITION_LR_INIT  = 0.00016      # multiplied by cameras_extent
POSITION_LR_FINAL = 0.0000016    # multiplied by cameras_extent
FEATURE_LR        = 0.0025       # SH band 0; bands 1..3 use FEATURE_LR / 20
OPACITY_LR        = 0.05
SCALING_LR        = 0.005
ROTATION_LR       = 0.001
LAMBDA_DSSIM      = 0.2
SH_DEGREE         = 3            # one band unlocked every 1000 iterations

### adaptive density control
DENSIFY_FROM           = 500
DENSIFY_UNTIL          = 15_000
DENSIFY_INTERVAL       = 100
DENSIFY_GRAD           = 0.0002  # threshold on the NDC screen-space gradient
PERCENT_DENSE          = 0.01    # clone/split boundary, times cameras_extent
OPACITY_RESET_INTERVAL = 3_000
MIN_OPACITY            = 0.005
MAX_SCREEN_SIZE        = 20      # px; only applied after OPACITY_RESET_INTERVAL
MAX_POINTS             = 0       # 0 = unlimited (paper); cap it on a small GPU

### bookkeeping
CACHE_DEVICE = "cpu"    # "gpu" caches training images in VRAM, "cpu" in pinned host RAM
LPIPS_NET    = "vgg"    # the paper reports VGG; alex reads 0.03-0.08 lower
LOG_EVERY    = 100
HIST_EVERY   = 10
SAVE_EVERY   = 2_000
RESULTS_DIR  = "./results"

C0 = 0.28209479177387814          # Y_00, for RGB <-> SH DC conversion
INV_SIG = lambda a: math.log(a / (1.0 - a))

# 3DGS @ 30k on mip-NeRF360 -> (PSNR, SSIM, LPIPS-VGG, #Gaussians)
PAPER = {
    "bicycle": (25.25, 0.771, 0.205, 6.1e6), "flowers":  (21.52, 0.605, 0.336, 3.6e6),
    "garden":  (27.41, 0.868, 0.103, 5.8e6), "stump":    (26.55, 0.775, 0.210, 4.9e6),
    "treehill":(22.49, 0.638, 0.317, 3.8e6), "room":     (30.63, 0.914, 0.220, 1.6e6),
    "counter": (28.70, 0.905, 0.204, 1.2e6), "kitchen":  (30.32, 0.922, 0.129, 1.8e6),
    "bonsai":  (31.98, 0.938, 0.205, 1.3e6),
}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"SCENE {SCENE} | ITERS {ITERATIONS} | DOWNSAMPLE {DOWNSAMPLE}")

## Step 1: The Data
### 1.1 Source

#### 1.1.1 Deep blending :
  *http://visual.cs.ucl.ac.uk/pubs/deepblending/*
  - Training data Training data (42.5 GB)
  - Test data Test data (33.5 GB) 

#### 1.1.2 mip-NeRF 360 :
  *https://jonbarron.info/mipnerf360/*
  - Dataset pt1: (11.7 GB):
  - Dataset pt2: (4.2 GB)

#### 1.1.3 Tanks&Temples :
  *https://www.tanksandtemples.org/download/*
  - Datasets,Intermediate,Advanced (>20 GB)


We are using the **1.1.2 mip-NeRF 360**

* **Download:** http://storage.googleapis.com/gresearch/refraw360/360_v2.zip  
* **wget:** `wget http://storage.googleapis.com/gresearch/refraw360/360_v2.zip`

### 1.2 Directory Structure

After extraction, ensure your folder looks like this:

```text
data/
  └── object|scene/
      ├── images/               # Images of the scene
      ├── images_2/             # downsampled by 2 Images of the scene
      ├── images_4/             # downsampled by 4 Images of the scene
      ├── images_8/             # downsampled by 8 Images of the scene
      └── sparse/0/             # COLMAP reconstruction
          ├── cameras.bin       # intrinsics
          ├── images.bin        # extrinsics, one entry per image
          └── points3D.bin      # sparse point cloud; initializes the Gaussian centers

cameras.bin (PINHOLE):
[fx, fy, cx, cy]

images.bin (per image):
[qw, qx, qy, qz, tx, ty, tz, camera_id, name]

    R = qvec2rot(q)     world -> camera rotation
    T = [tx, ty, tz]    world -> camera translation
    camera center = -R^T . T

points3D.bin (per point):
[id, x, y, z, r, g, b, error, track]
```

We read `sparse/0/` directly and ignore `poses_bounds.npy`.
The LLFF format is lossy in three ways that all cost PSNR:

    one shared focal length     ->  fx and fy forced equal
    principal point assumed at  ->  (W/2, H/2)
    camera basis in the columns ->  [down, right, backwards], needs a hand-rolled permutation

COLMAP already stores world->camera in the convention the rasterizer wants
(x right, y down, z forward), and images.bin / points3D.bin share one frame
by construction, so there is no conversion left to get wrong.

## Step 2: Dataset and DataLoader

### Step 2.1 : Dataset

In [ ]:
_CAM_MODELS = {0: ("SIMPLE_PINHOLE", 3), 1: ("PINHOLE", 4), 2: ("SIMPLE_RADIAL", 4),
               3: ("RADIAL", 5), 4: ("OPENCV", 8), 5: ("OPENCV_FISHEYE", 8),
               6: ("FULL_OPENCV", 12), 7: ("FOV", 5), 8: ("SIMPLE_RADIAL_FISHEYE", 4),
               9: ("RADIAL_FISHEYE", 5), 10: ("THIN_PRISM_FISHEYE", 12)}


def qvec2rot(w, x, y, z):
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*w*z,     2*x*z + 2*w*y],
        [2*x*y + 2*w*z,     1 - 2*x*x - 2*z*z, 2*y*z - 2*w*x],
        [2*x*z - 2*w*y,     2*y*z + 2*w*x,     1 - 2*x*x - 2*y*y]])


def read_colmap_cameras(base):
    cams = {}
    with open(os.path.join(base, "cameras.bin"), "rb") as f:
        for _ in range(struct.unpack("<Q", f.read(8))[0]):
            cid, mid, W, H = struct.unpack("<iiQQ", f.read(24))
            name, npar = _CAM_MODELS[mid]
            par = struct.unpack("<" + "d" * npar, f.read(8 * npar))
            if name == "PINHOLE":
                fx, fy, cx, cy = par
            elif name in ("SIMPLE_PINHOLE", "SIMPLE_RADIAL", "RADIAL", "SIMPLE_RADIAL_FISHEYE"):
                fx = fy = par[0]; cx, cy = par[1], par[2]
            else:
                fx, fy, cx, cy = par[:4]
            if npar > 4 and any(abs(v) > 1e-6 for v in par[4:]):
                print(f"  WARNING camera {cid} ({name}) has nonzero distortion; "
                      f"use the undistorted images/ folder")
            cams[cid] = dict(W=W, H=H, fx=fx, fy=fy, cx=cx, cy=cy)
    return cams


def read_colmap_images(base):
    out = []
    with open(os.path.join(base, "images.bin"), "rb") as f:
        for _ in range(struct.unpack("<Q", f.read(8))[0]):
            iid, qw, qx, qy, qz, tx, ty, tz, cid = struct.unpack("<idddddddi", f.read(64))
            nm = b""
            while (c := f.read(1)) != b"\x00":
                nm += c
            f.seek(struct.unpack("<Q", f.read(8))[0] * 24, 1)   # skip the points2D block
            w2c = np.eye(4, dtype=np.float64)
            w2c[:3, :3] = qvec2rot(qw, qx, qy, qz)              # already world -> camera
            w2c[:3, 3] = (tx, ty, tz)
            out.append((nm.decode(), cid, w2c))
    return sorted(out, key=lambda r: r[0])


class MyDataset:
    def __init__(self, scene=SCENE, downsample=DOWNSAMPLE):
        base = os.path.join("data", scene, "sparse", "0")
        cams = read_colmap_cameras(base)
        entries = read_colmap_images(base)

        img_dir = os.path.join("data", scene, "images" if downsample == 1 else f"images_{downsample}")
        avail = set(os.listdir(img_dir))

        # match poses to files by NAME, so an unregistered image cannot silently
        # shift every subsequent pose by one
        self.image_paths, self.w2c, intr = [], [], []
        for name, cid, w2c in entries:
            fn = name if name in avail else None
            if fn is None:
                stem = os.path.splitext(name)[0]
                fn = next((a for a in avail if os.path.splitext(a)[0] == stem), None)
                if fn is None:
                    continue
            self.image_paths.append(os.path.join(img_dir, fn))
            self.w2c.append(w2c.astype(np.float32))
            intr.append(cams[cid])
        assert self.image_paths, f"no images matched between {img_dir} and images.bin"

        # rescale intrinsics by the ACTUAL images_N/ size: COLMAP rounds when it
        # writes those folders, so f/downsample drifts by a fraction of a pixel
        with Image.open(self.image_paths[0]) as im:
            self.W, self.H = im.size
        c0 = intr[0]
        sx, sy = self.W / c0["W"], self.H / c0["H"]
        self.cam = [dict(fx=c["fx"] * sx, fy=c["fy"] * sy,
                         cx=c["cx"] * sx, cy=c["cy"] * sy) for c in intr]

        # getNerfppNorm: the scene stays in COLMAP units and the camera radius
        # is the spatial_lr_scale for the position learning rate
        centers = np.stack([-w[:3, :3].T @ w[:3, 3] for w in self.w2c])
        self.cameras_extent = float(np.linalg.norm(centers - centers.mean(0), axis=1).max() * 1.1)

        n = len(self.image_paths)
        self.test_idx = [i for i in range(n) if i % TEST_EVERY == 0]
        self.train_idx = [i for i in range(n) if i % TEST_EVERY != 0]

        k = self.cam[0]
        print(f"  intrinsics: fx={k['fx']:.2f} fy={k['fy']:.2f} (fy/fx={k['fy']/k['fx']:.5f})"
              f"  cx={k['cx']:.2f} (W/2={self.W/2:.1f}) cy={k['cy']:.2f} (H/2={self.H/2:.1f})")
        print(f"  cameras_extent = {self.cameras_extent:.4f}")
        print(f"{scene}: {len(self.train_idx)} train / {len(self.test_idx)} test views "
              f"at {self.W}x{self.H}")

    def __len__(self):
        return len(self.image_paths)

    def load(self, idx, dev):
        """dev=None keeps the image in pinned host memory for an async H2D copy."""
        with Image.open(self.image_paths[idx]) as im:
            arr = np.array(im.convert("RGB"))
        img = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
        img = img.to(dev) if dev is not None else img.pin_memory()
        w2c = torch.from_numpy(self.w2c[idx]).to(device)
        return img, (w2c, self.cam[idx])

### Step 2.2: Guassian Model

In [ ]:
class GaussianModel(nn.Module):
    # sh is split into DC / rest so the two can carry different learning rates
    PARAMS = ("xyz", "sh_dc", "sh_rest", "opacity", "S", "q")

    def __init__(self, scene=SCENE, empty=False):
        super().__init__()
        self.active_sh_degree = 0
        self.max_sh_degree = SH_DEGREE
        if empty:
            return

        path = f"data/{scene}/sparse/0/points3D.bin"
        chunk_fmt = "<QdddBBBdQ"
        chunk_size = struct.calcsize(chunk_fmt)

        with open(path, "rb") as f:
            n = struct.unpack("<Q", f.read(8))[0]
            xyz = np.empty((n, 3), dtype=np.float64)
            rgb = np.empty((n, 3), dtype=np.uint8)
            for i in range(n):
                pid, x, y, z, r, g, b, err, track_len = struct.unpack(chunk_fmt, f.read(chunk_size))
                xyz[i] = (x, y, z)
                rgb[i] = (r, g, b)
                f.seek(track_len * 8, 1)

        xyz = torch.tensor(xyz, dtype=torch.float32)

        # ----- calc S
        # distCUDA2 is the mean SQUARED distance to the 3 nearest neighbours,
        # then sqrt -- the RMS, not the arithmetic mean
        nn_finder = NearestNeighbors(n_neighbors=4, metric="euclidean").fit(xyz)
        dists, _ = nn_finder.kneighbors(xyz)
        rms_dist = torch.from_numpy(np.sqrt((dists[:, 1:] ** 2).mean(axis=1))).float()
        self.S = nn.Parameter(torch.log(rms_dist.clamp(min=1e-7)).unsqueeze(1).repeat(1, 3))

        # ----- calc q
        q = torch.zeros(n, 4); q[:, 0] = 1.0
        self.q = nn.Parameter(q)

        # ----- calc o
        self.opacity = nn.Parameter(torch.full((n, 1), INV_SIG(0.1)))

        # ----- calc location(xyz)
        self.xyz = nn.Parameter(xyz.requires_grad_(True))

        # ------ color params:
        rgb01 = torch.tensor(rgb, dtype=torch.float32) / 255.0
        self.sh_dc = nn.Parameter(((rgb01 - 0.5) / C0).unsqueeze(1))   # (N,1,3)
        self.sh_rest = nn.Parameter(torch.zeros(n, 15, 3))             # (N,15,3)

    @property
    def sh(self):
        return torch.cat([self.sh_dc, self.sh_rest], dim=1)            # (N,16,3)

    @property
    def N(self):
        return self.xyz.shape[0]

    def oneup_sh_degree(self):
        if self.active_sh_degree < self.max_sh_degree:
            self.active_sh_degree += 1

    def build_rotation(self):
        q = self.q / (self.q.norm(dim=1, keepdim=True) + 1e-12)
        r, i, j, k = q.unbind(dim=1)
        R00 = 1 - 2*(j*j + k*k); R01 = 2*(i*j - r*k);     R02 = 2*(i*k + r*j)
        R10 = 2*(i*j + r*k);     R11 = 1 - 2*(i*i + k*k); R12 = 2*(j*k - r*i)
        R20 = 2*(i*k - r*j);     R21 = 2*(j*k + r*i);     R22 = 1 - 2*(i*i + j*j)
        return torch.stack([
            torch.stack([R00, R01, R02], dim=1),
            torch.stack([R10, R11, R12], dim=1),
            torch.stack([R20, R21, R22], dim=1),
        ], dim=1)

    def get_cov(self):
        R = self.build_rotation()
        S = torch.diag_embed(torch.exp(self.S))
        return R @ S @ S.transpose(1, 2) @ R.transpose(1, 2)

    # ---- optimizer-aware parameter surgery ---------------------------------
    # Densification must carry the Adam moments (exp_avg, exp_avg_sq) through
    # concat / prune / reset. Rebuilding the optimizer instead would wipe them
    # 145 times over a 30k run.
    def _cat_to_optimizer(self, optimizer, new):
        for g in optimizer.param_groups:
            ext = new[g["name"]]
            old = g["params"][0]
            st = optimizer.state.get(old, None)
            if st is not None:
                del optimizer.state[old]
                st["exp_avg"] = torch.cat([st["exp_avg"], torch.zeros_like(ext)], 0)
                st["exp_avg_sq"] = torch.cat([st["exp_avg_sq"], torch.zeros_like(ext)], 0)
            p = nn.Parameter(torch.cat([old.data, ext], 0).requires_grad_(True))
            g["params"][0] = p
            if st is not None:
                optimizer.state[p] = st
            setattr(self, g["name"], p)

    def _prune_optimizer(self, optimizer, valid):
        for g in optimizer.param_groups:
            old = g["params"][0]
            st = optimizer.state.get(old, None)
            if st is not None:
                del optimizer.state[old]
                st["exp_avg"] = st["exp_avg"][valid]
                st["exp_avg_sq"] = st["exp_avg_sq"][valid]
            p = nn.Parameter(old.data[valid].requires_grad_(True))
            g["params"][0] = p
            if st is not None:
                optimizer.state[p] = st
            setattr(self, g["name"], p)

    def _replace_in_optimizer(self, optimizer, name, tensor):
        for g in optimizer.param_groups:
            if g["name"] != name:
                continue
            old = g["params"][0]
            st = optimizer.state.get(old, None)
            if st is not None:
                del optimizer.state[old]
                st["exp_avg"] = torch.zeros_like(tensor)
                st["exp_avg_sq"] = torch.zeros_like(tensor)
            p = nn.Parameter(tensor.requires_grad_(True))
            g["params"][0] = p
            if st is not None:
                optimizer.state[p] = st
            setattr(self, name, p)

    # ---- adaptive density control ------------------------------------------
    def clone_gaussians(self, optimizer, mask, stats):
        self._cat_to_optimizer(optimizer, {
            "xyz": self.xyz[mask], "S": self.S[mask], "q": self.q[mask],
            "opacity": self.opacity[mask],
            "sh_dc": self.sh_dc[mask], "sh_rest": self.sh_rest[mask],
        })
        stats.grow(int(mask.sum()))

    def split_gaussians(self, optimizer, mask, stats, n_split=2):
        """Split large Gaussians into two smaller ones."""
        stds = torch.exp(self.S[mask]).repeat(n_split, 1)
        samples = torch.normal(mean=torch.zeros_like(stds), std=stds)
        rots = self.build_rotation()[mask].repeat(n_split, 1, 1)
        perturb = torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1)

        self._cat_to_optimizer(optimizer, {
            "xyz": self.xyz[mask].repeat(n_split, 1) + perturb,
            "S": self.S[mask].repeat(n_split, 1) - math.log(0.8 * n_split),
            "q": self.q[mask].repeat(n_split, 1),
            "opacity": self.opacity[mask].repeat(n_split, 1),
            "sh_dc": self.sh_dc[mask].repeat(n_split, 1, 1),
            "sh_rest": self.sh_rest[mask].repeat(n_split, 1, 1),
        })
        stats.grow(int(mask.sum()) * n_split)
        # the parents are deleted, not merely made transparent
        kill = torch.cat([mask, torch.zeros(int(mask.sum()) * n_split,
                                            dtype=torch.bool, device=mask.device)])
        self.prune_points(optimizer, kill, stats)

    def prune_points(self, optimizer, kill_mask, stats):
        valid = ~kill_mask
        self._prune_optimizer(optimizer, valid)
        stats.filter(valid)

    def densify_and_prune(self, optimizer, stats, extent, max_screen=None):
        grads = (stats.grad_accum / stats.denom.clamp(min=1)).squeeze(-1)
        grads[stats.denom.squeeze(-1) == 0] = 0.0
        scales = torch.exp(self.S).max(dim=1).values

        n_cloned = n_split = 0
        if not MAX_POINTS or self.N < MAX_POINTS:
            clone_mask = (grads >= DENSIFY_GRAD) & (scales <= PERCENT_DENSE * extent)
            n_cloned = int(clone_mask.sum())
            self.clone_gaussians(optimizer, clone_mask, stats)

            # the clones just appended cannot themselves split this round
            grads2 = torch.zeros(self.N, device=grads.device)
            grads2[: grads.shape[0]] = grads
            scales = torch.exp(self.S).max(dim=1).values
            split_mask = (grads2 >= DENSIFY_GRAD) & (scales > PERCENT_DENSE * extent)
            n_split = int(split_mask.sum())
            self.split_gaussians(optimizer, split_mask, stats)

        # FIX(22): official's densification_postfix zeroes xyz_gradient_accum,
        # denom AND max_radii2D, and it runs INSIDE densify_and_clone /
        # densify_and_split -- i.e. BEFORE prune_mask is built. So
        # `big_points_vs = self.max_radii2D > max_screen_size` is always
        # evaluated against an all-zero tensor and the screen-size prune never
        # fires. It is dead code in the reference implementation, and the
        # published numbers come from that behaviour.
        # I kept the accumulated maxima alive until the end of the function, so
        # mine fired at full strength: ~30k Gaussians culled every 100 iterations
        # from iteration 3000 on, which cancelled all clone/split growth and
        # capped the model at ~900k instead of ~1.25M.
        stats.postfix()

        scales = torch.exp(self.S).max(dim=1).values
        n_before = self.N
        kill = (torch.sigmoid(self.opacity) < MIN_OPACITY).squeeze(-1)
        n_op = int(kill.sum())
        # both size prunes are gated on max_screen, which train() only passes
        # after OPACITY_RESET_INTERVAL -- culling large low-gradient Gaussians
        # earlier removes exactly what should be covering smooth regions
        n_ws = n_vs = 0
        if max_screen is not None:
            big_ws = scales > 0.1 * extent
            big_vs = stats.max_radii2D > max_screen
            n_ws, n_vs = int(big_ws.sum()), int(big_vs.sum())
            kill |= big_ws
            kill |= big_vs
        self.prune_points(optimizer, kill, stats)
        return dict(before=n_before, after=self.N, cloned=n_cloned, split=n_split,
                    pruned=int(kill.sum()), p_op=n_op, p_ws=n_ws, p_vs=n_vs)

    def reset_opacity(self, optimizer):
        # min(), not fill_(): the paper only CAPS opacity so that floaters have
        # to re-earn their alpha; overwriting it destroys every converged Gaussian
        capped = torch.min(self.opacity.data,
                           torch.full_like(self.opacity.data, INV_SIG(0.01)))
        self._replace_in_optimizer(optimizer, "opacity", capped)


class DensifyStats:
    """Per-Gaussian running stats, kept in lockstep with the parameter tensors."""

    def __init__(self, n, dev):
        self.dev = dev
        self.grad_accum = torch.zeros(n, 1, device=dev)
        self.denom = torch.zeros(n, 1, device=dev)
        self.max_radii2D = torch.zeros(n, device=dev)

    def add(self, means2D_grad, visible):
        # DENSIFY_GRAD is calibrated on the NDC screen-space gradient of the 2D
        # means, not on |grad(xyz)| in world units
        self.grad_accum[visible] += means2D_grad[visible, :2].norm(dim=-1, keepdim=True)
        self.denom[visible] += 1

    def grow(self, k):
        z1 = torch.zeros(k, 1, device=self.dev)
        self.grad_accum = torch.cat([self.grad_accum, z1])
        self.denom = torch.cat([self.denom, z1.clone()])
        self.max_radii2D = torch.cat([self.max_radii2D, torch.zeros(k, device=self.dev)])

    def filter(self, valid):
        self.grad_accum = self.grad_accum[valid]
        self.denom = self.denom[valid]
        self.max_radii2D = self.max_radii2D[valid]

    def postfix(self):
        """Mirror of the official densification_postfix: zero all three running
        stats. Called after clone/split and BEFORE the prune mask is built."""
        self.grad_accum.zero_(); self.denom.zero_(); self.max_radii2D.zero_()

## Step 3 : Optimization

In [ ]:
def get_projection_matrix(W, H, fx, fy, cx, cy, znear=0.01, zfar=100.0):
    """The rasterizer's own convention: z_sign = +1, P[3,2] = +1.

    The OpenGL matrix (P[3,2] = -1) gives w = -z, so the perspective divide
    negates NDC x and y and the render comes out mirrored through the centre.
    P[0,2]/P[1,2] carry the principal point: ndc_x = 2*(fx*x/z + cx)/W - 1.
    """
    P = torch.zeros(4, 4)
    P[0, 0] = 2 * fx / W
    P[1, 1] = 2 * fy / H
    P[0, 2] = 2 * cx / W - 1
    P[1, 2] = 2 * cy / H - 1
    P[2, 2] = zfar / (zfar - znear)
    P[2, 3] = -(zfar * znear) / (zfar - znear)
    P[3, 2] = 1.0
    return P


# newer diff-gaussian-rasterization builds added an `antialiasing` field
_RS_HAS_AA = "antialiasing" in getattr(GaussianRasterizationSettings, "_fields", ())


def make_rasterizer(H, W, k, w2c, cam_pos, sh_degree, bg):
    proj = get_projection_matrix(W, H, k["fx"], k["fy"], k["cx"], k["cy"]).to(w2c.device)
    kw = dict(
        image_height=int(H), image_width=int(W),
        tanfovx=0.5 * W / k["fx"], tanfovy=0.5 * H / k["fy"],
        bg=bg, scale_modifier=1.0,
        viewmatrix=w2c.transpose(0, 1),
        projmatrix=(proj @ w2c).transpose(0, 1),
        sh_degree=sh_degree, campos=cam_pos, prefiltered=False, debug=False,
    )
    if _RS_HAS_AA:
        kw["antialiasing"] = False
    return GaussianRasterizer(GaussianRasterizationSettings(**kw))


def render(model, H, W, k, w2c, bg):
    cam_pos = torch.linalg.inv(w2c)[:3, 3]
    means2D = torch.zeros_like(model.xyz, requires_grad=True)
    means2D.retain_grad()
    rasterizer = make_rasterizer(H, W, k, w2c, cam_pos, model.active_sh_degree, bg)
    out = rasterizer(
        means3D=model.xyz,
        means2D=means2D,
        shs=model.sh,
        colors_precomp=None,
        opacities=torch.sigmoid(model.opacity),
        scales=torch.exp(model.S),
        # the quaternion MUST be normalized here. In forward.cu the normalization
        # inside computeCov3D is commented out:
        #     glm::vec4 q = rot;// / glm::length(rot);
        # so a non-unit q does not merely rotate -- the quaternion-to-matrix
        # formula yields R scaled by |q|^2, i.e. covariance scaled by |q|^4.
        rotations=F.normalize(model.q, dim=1),
        cov3D_precomp=None,
    )
    image, radii = out[0], out[1]
    return image, radii, means2D


def get_expon_lr_func(lr_init, lr_final, max_steps):
    def helper(step):
        t = np.clip(step / max_steps, 0, 1)
        return float(np.exp(np.log(lr_init) * (1 - t) + np.log(lr_final) * t))
    return helper

In [ ]:
def train():
    ds = MyDataset(SCENE, downsample=DOWNSAMPLE)
    cache_dev = device if CACHE_DEVICE == "gpu" else None
    cached = [ds.load(i, cache_dev) for i in ds.train_idx]

    model = GaussianModel(SCENE).to(device)
    print(f"Initialised {model.N} Gaussians")

    extent = ds.cameras_extent
    optimizer = optim.Adam([
        {"params": [model.xyz],     "lr": POSITION_LR_INIT * extent, "name": "xyz"},
        {"params": [model.sh_dc],   "lr": FEATURE_LR,                "name": "sh_dc"},
        {"params": [model.sh_rest], "lr": FEATURE_LR / 20.0,         "name": "sh_rest"},
        {"params": [model.opacity], "lr": OPACITY_LR,                "name": "opacity"},
        {"params": [model.S],       "lr": SCALING_LR,                "name": "S"},
        {"params": [model.q],       "lr": ROTATION_LR,               "name": "q"},
    ], lr=0.0, eps=1e-15, betas=(0.9, 0.999))

    xyz_lr = get_expon_lr_func(POSITION_LR_INIT * extent,
                               POSITION_LR_FINAL * extent, ITERATIONS)
    stats = DensifyStats(model.N, device)
    bg = torch.zeros(3, device=device)

    history = {"iter": [], "l1": [], "dssim": [], "loss": [], "psnr": [], "points": []}
    order = []
    start = time.time()
    last_t, last_it = start, 0

    for it in range(1, ITERATIONS + 1):
        for g in optimizer.param_groups:
            if g["name"] == "xyz":
                g["lr"] = xyz_lr(it)
        if it % 1000 == 0:
            model.oneup_sh_degree()

        # sample without replacement, reshuffling once the epoch is exhausted
        if not order:
            order = list(range(len(cached)))
            random.shuffle(order)
        img_u8, (w2c, k) = cached[order.pop()]
        gt = img_u8.to(device, non_blocking=True).float() / 255.0

        image, radii, means2D = render(model, ds.H, ds.W, k, w2c, bg)

        l1 = torch.abs(image - gt).mean()
        dssim = 1.0 - ssim_loss(image, gt)
        loss = (1.0 - LAMBDA_DSSIM) * l1 + LAMBDA_DSSIM * dssim

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        with torch.no_grad():
            if it < DENSIFY_UNTIL:
                vis = radii > 0
                stats.max_radii2D[vis] = torch.max(stats.max_radii2D[vis], radii[vis].float())
                stats.add(means2D.grad, vis)

        optimizer.step()

        with torch.no_grad():
            if DENSIFY_FROM <= it <= DENSIFY_UNTIL and it % DENSIFY_INTERVAL == 0:
                d = model.densify_and_prune(
                    optimizer, stats, extent,
                    max_screen=MAX_SCREEN_SIZE if it > OPACITY_RESET_INTERVAL else None,
                )
                torch.cuda.empty_cache()
                if it % 1000 == 0:
                    print(f"  [densify {it}] {d['before']} -> {d['after']}  "
                          f"+{d['cloned']} clone +{d['split']} split  "
                          f"-{d['pruned']} prune (op {d['p_op']}, ws {d['p_ws']}, vs {d['p_vs']})")

            # only while densification is active, so pruning can still clean up
            if it % OPACITY_RESET_INTERVAL == 0 and it < DENSIFY_UNTIL:
                model.reset_opacity(optimizer)

            if it % HIST_EVERY == 0:
                mse = torch.mean((image - gt) ** 2).item()
                history["iter"].append(it)
                history["l1"].append(l1.item())
                history["dssim"].append(dssim.item())
                history["loss"].append(loss.item())
                history["psnr"].append(-10.0 * math.log10(max(mse, 1e-12)))
                history["points"].append(model.N)

            if SAVE_EVERY and it % SAVE_EVERY == 0:
                torch.save({"params": {p: getattr(model, p).data for p in GaussianModel.PARAMS},
                            "active_sh_degree": model.active_sh_degree, "iteration": it},
                           f"{RESULTS_DIR}/ckpt_{SCENE}_latest.pth")

            if it % LOG_EVERY == 0:
                now = time.time()
                rate = (now - last_t) / max(it - last_it, 1)
                last_t, last_it = now, it
                eta = rate * (ITERATIONS - it)
                print(f"Iter {it}/{ITERATIONS} | L1 {history['l1'][-1]:.4f} | "
                      f"PSNR {history['psnr'][-1]:.2f} | Pts {model.N} | "
                      f"{1 / rate:5.1f} it/s | "
                      f"Mem {torch.cuda.max_memory_allocated() / 1024 ** 3:.2f}GB peak | "
                      f"ETA {time.strftime('%H:%M:%S', time.gmtime(eta))}")

    ckpt_path = f"{RESULTS_DIR}/gaussian_model_{SCENE}_{ITERATIONS}.pth"
    torch.save({"params": {p: getattr(model, p).data for p in GaussianModel.PARAMS},
                "active_sh_degree": model.active_sh_degree}, ckpt_path)
    print(f"\n[TRAINING COMPLETE] {model.N} Gaussians -> {ckpt_path}")
    np.save(f"{RESULTS_DIR}/history_{SCENE}_{ITERATIONS}.npy", history, allow_pickle=True)

    del cached, model, optimizer, stats
    gc.collect(); torch.cuda.empty_cache()
    return ckpt_path, history

In [ ]:
ckpt_path, history = train()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
it = history["iter"]
resets = [i for i in range(OPACITY_RESET_INTERVAL, DENSIFY_UNTIL, OPACITY_RESET_INTERVAL)]

def mark(a):
    for r in resets:
        a.axvline(r, color="crimson", ls=":", lw=1, alpha=0.7)
    a.axvline(DENSIFY_UNTIL, color="k", ls="--", lw=1, alpha=0.7)
    a.grid(alpha=0.25)

def smooth(y, w=25):
    return np.convolve(y, np.ones(w) / w, mode="valid")

ax[0, 0].plot(it, history["loss"], lw=0.6, alpha=0.35, color="tab:blue")
ax[0, 0].plot(it[len(it) - len(smooth(history["loss"])):], smooth(history["loss"]),
              lw=1.6, color="tab:blue", label="0.8*L1 + 0.2*D-SSIM")
ax[0, 0].plot(it[len(it) - len(smooth(history["l1"])):], smooth(history["l1"]),
              lw=1.4, color="tab:orange", label="L1")
ax[0, 0].set_yscale("log"); ax[0, 0].set_title("loss"); ax[0, 0].legend(fontsize=8)

ax[0, 1].plot(it, history["psnr"], lw=0.6, alpha=0.3, color="tab:green")
ax[0, 1].plot(it[len(it) - len(smooth(history["psnr"])):], smooth(history["psnr"]),
              lw=1.8, color="tab:green")
if SCENE in PAPER:
    ax[0, 1].axhline(PAPER[SCENE][0], color="tab:red", lw=1.2,
                     label=f"paper test {PAPER[SCENE][0]:.2f}")
    ax[0, 1].legend(fontsize=8)
ax[0, 1].set_title("train PSNR (dB)")

ax[1, 0].plot(it, history["points"], lw=1.5, color="tab:purple")
if SCENE in PAPER:
    ax[1, 0].axhline(PAPER[SCENE][3], color="tab:red", lw=1.2,
                     label=f"paper ~{PAPER[SCENE][3]/1e6:.1f}M")
    ax[1, 0].legend(fontsize=8)
ax[1, 0].set_title("number of Gaussians")

ax[1, 1].plot(it[len(it) - len(smooth(history["dssim"])):], smooth(history["dssim"]),
              lw=1.6, color="tab:brown")
ax[1, 1].set_yscale("log"); ax[1, 1].set_title("D-SSIM")

for a in ax.ravel():
    mark(a); a.set_xlabel("iteration")
fig.suptitle(f"{SCENE}  |  dotted red = opacity reset,  dashed black = densification ends", y=1.0)
fig.tight_layout()
plt.show()

## Step 4 : Evaluation

In [ ]:
@torch.no_grad()
def evaluate(ckpt_path):
    """Score the held-out every-8th views at the training resolution."""
    ds = MyDataset(SCENE, downsample=DOWNSAMPLE)

    model = GaussianModel(empty=True)
    ckpt = torch.load(ckpt_path, map_location=device)
    for name, v in ckpt["params"].items():
        setattr(model, name, nn.Parameter(v.to(device)))
    model.active_sh_degree = ckpt["active_sh_degree"]
    model.to(device)
    print(f"Loaded {model.N} points, sh_degree={model.active_sh_degree}")

    lpips_fn = lpips.LPIPS(net=LPIPS_NET).to(device)
    bg = torch.zeros(3, device=device)
    psnrs, ssims, lps, lps_repo, renders, gts = [], [], [], [], [], []

    for i in tqdm(ds.test_idx, desc="Rendering", unit="frame", colour="green"):
        img_u8, (w2c, k) = ds.load(i, device)
        gt = img_u8.float() / 255.0
        image, _, _ = render(model, ds.H, ds.W, k, w2c, bg)
        image = image.clamp(0, 1)                      # official clamps before scoring

        psnrs.append((-10 * torch.log10(torch.mean((image - gt) ** 2))).item())
        ssims.append(ssim(image.unsqueeze(0), gt.unsqueeze(0),
                          data_range=1.0, size_average=True).item())
        # FIX(23): the official metrics.py does NOT use the lpips pip package --
        # it uses its own lpipsPyTorch, whose BaseNet.normalize applies the
        # standard LPIPS constants
        #     mean = [-.030,-.088,-.188]   std = [.458,.448,.450]
        # which are defined for inputs in [-1,1]. But metrics.py feeds it
        # tf.to_tensor(Image.open(...)), i.e. [0,1]. Halving the effective input
        # range shrinks the feature differences and pushes LPIPS down, so their
        # published number is not comparable to a correctly-scaled one.
        # Report both: `std` is the correct usage, `3dgs` matches their table.
        lps.append(lpips_fn(image.unsqueeze(0) * 2 - 1, gt.unsqueeze(0) * 2 - 1).item())
        lps_repo.append(lpips_fn(image.unsqueeze(0), gt.unsqueeze(0)).item())   # FIX(23)
        renders.append((image.permute(1, 2, 0) * 255).cpu().numpy().astype(np.uint8))
        gts.append((gt.permute(1, 2, 0) * 255).cpu().numpy().astype(np.uint8))

    metrics = dict(psnr=float(np.mean(psnrs)), ssim=float(np.mean(ssims)),
                   lpips=float(np.mean(lps)), lpips_repo=float(np.mean(lps_repo)),
                   points=model.N, per_view_psnr=psnrs)

    # metrics first: a codec failure must never destroy a completed run
    print("-" * 46)
    print(f"scene {SCENE} | {model.N} Gaussians | {len(ds.test_idx)} test views @ {ds.W}x{ds.H}")
    if SCENE in PAPER:
        p_psnr, p_ssim, p_lpips, p_pts = PAPER[SCENE]
        print(f"{'':6s}{'ours':>10s}{'paper':>10s}{'delta':>10s}")
        print(f"{'PSNR':6s}{metrics['psnr']:>10.3f}{p_psnr:>10.2f}{metrics['psnr']-p_psnr:>+10.2f}")
        print(f"{'SSIM':6s}{metrics['ssim']:>10.4f}{p_ssim:>10.3f}{metrics['ssim']-p_ssim:>+10.3f}")
        print(f"{'LPIPS':6s}{metrics['lpips']:>10.4f}{'':>20s}  [-1,1] std")
        print(f"{'LPIPS':6s}{metrics['lpips_repo']:>10.4f}{p_lpips:>10.3f}"
              f"{metrics['lpips_repo']-p_lpips:>+10.3f}  [0,1] 3dgs-repo")
        print(f"{'pts':6s}{model.N:>10d}{int(p_pts):>10d}{model.N-int(p_pts):>+10d}")
    else:
        print(f"PSNR  {metrics['psnr']:.3f}   SSIM {metrics['ssim']:.4f}   "
              f"LPIPS {metrics['lpips']:.4f} / {metrics['lpips_repo']:.4f} [{LPIPS_NET}]")
    print("-" * 46)

    with open(f"{RESULTS_DIR}/metrics_{SCENE}_{ITERATIONS}.txt", "w") as fh:
        fh.write(f"scene {SCENE}\npoints {model.N}\nres {ds.W}x{ds.H}\n"
                 f"psnr {metrics['psnr']:.4f}\nssim {metrics['ssim']:.4f}\n"
                 f"lpips_{LPIPS_NET}_std {metrics['lpips']:.4f}\n"
                 f"lpips_{LPIPS_NET}_3dgs {metrics['lpips_repo']:.4f}\n")

    video_path = f"{RESULTS_DIR}/render_{SCENE}_{ITERATIONS}.mp4"
    try:
        eh, ew = (ds.H // 2) * 2, (ds.W // 2) * 2     # h264 + yuv420p needs even dims
        imageio.mimwrite(video_path, [f[:eh, :ew] for f in renders],
                         fps=10, quality=8, macro_block_size=1)
        print(f"Video {video_path}")
    except Exception as e:
        print(f"(video encode failed, metrics above are unaffected: {e})")

    return metrics, renders, gts, video_path


metrics, renders, gts, video_path = evaluate(ckpt_path)

In [ ]:
### show: ground truth | render | absolute error, for the best / median / worst test views
order = np.argsort(metrics["per_view_psnr"])
picks = [order[-1], order[len(order) // 2], order[0]]
tags = ["best", "median", "worst"]

fig, ax = plt.subplots(len(picks), 3, figsize=(15, 4.2 * len(picks)))
for row, (idx, tag) in enumerate(zip(picks, tags)):
    gt, rd = gts[idx], renders[idx]
    err = np.abs(gt.astype(np.int16) - rd.astype(np.int16)).mean(-1)
    ax[row, 0].imshow(gt);  ax[row, 0].set_title(f"{tag}: ground truth")
    ax[row, 1].imshow(rd);  ax[row, 1].set_title(f"render  {metrics['per_view_psnr'][idx]:.2f} dB")
    im = ax[row, 2].imshow(err, cmap="inferno", vmin=0, vmax=60)
    ax[row, 2].set_title("|error|")
    fig.colorbar(im, ax=ax[row, 2], fraction=0.032)
    for a in ax[row]:
        a.axis("off")
fig.suptitle(f"{SCENE} @ {ITERATIONS} iters -- PSNR {metrics['psnr']:.2f} / "
             f"SSIM {metrics['ssim']:.4f} / LPIPS {metrics['lpips_repo']:.4f} "
             f"(3dgs convention)", y=1.0)
fig.tight_layout()
plt.show()

### per-view PSNR spread: a tight spread means the cameras are registered well
plt.figure(figsize=(13, 3))
plt.bar(range(len(metrics["per_view_psnr"])), metrics["per_view_psnr"], color="tab:blue")
plt.axhline(metrics["psnr"], color="k", lw=1, label=f"mean {metrics['psnr']:.2f}")
if SCENE in PAPER:
    plt.axhline(PAPER[SCENE][0], color="tab:red", lw=1.2, label=f"paper {PAPER[SCENE][0]:.2f}")
plt.xlabel("test view"); plt.ylabel("PSNR (dB)"); plt.legend(fontsize=8)
plt.grid(alpha=0.25, axis="y"); plt.tight_layout(); plt.show()

from IPython.display import Video
Video(video_path, embed=True, width=900)